# Notebook 06 :Tableaux de synthèse descriptif

---

**Entrée** : `data/processed/cohort_nlp.csv`

**Objectifs** :

1. Générer une **Table 1 au format revue scientifique** avec les principales caractéristiques cliniques.
2. Générer une **table exhaustive intégrant toutes les variables présentes dans `cohort_nlp.csv`**.

**Principe méthodologique** :

- les statistiques descriptives sont calculées sur les valeurs observées avant imputation ;
- aucune transformation utilisée pour le modèle n’est réapprise ici ;
- la table exhaustive inclut toutes les colonnes sources de `cohort_nlp.csv`, y compris les identifiants, variables temporelles, variables cliniques, comorbidités, variables textuelles et variables de sortie ;
- `charlson_score` est utilisé en priorité lorsqu’il est présent.

**Sorties principales** :

- `results/tables/tab39_table1_format_revue.csv`
- `results/tables/tab39_table1_format_revue.xlsx`
- `results/tables/table_variables_completes_cohort_nlp.csv`
- `results/tables/table_variables_completes_cohort_nlp.xlsx`


In [1]:
# ── Configuration et chargement de la cohorte ─────────────────────────────
# Le notebook fonctionne s'il est lancé depuis /notebooks ou depuis la racine du projet.

from pathlib import Path
import pandas as pd
import numpy as np
import re

#  Chemins projet — bloc unifié, identique dans NB01 -> NB08
def _racine_projet(reperes=("data", "results", "models", "notebooks")):
    """Racine = 1er dossier (courant ou parent) contenant un dossier repere."""
    for _p in (Path.cwd(), *Path.cwd().parents):
        if any((_p / _m).is_dir() for _m in reperes):
            return _p
    return Path.cwd()

ROOT_DIR   = _racine_projet()
BASE       = ROOT_DIR / "data"
OUT_DIR    = ROOT_DIR / "data" / "processed"
SPLIT_DIR  = OUT_DIR / "splits"
FIG_DIR    = ROOT_DIR / "figures"
RESULT_DIR = ROOT_DIR / "results"
TABLE_DIR  = RESULT_DIR / "tables"
MODEL_DIR  = ROOT_DIR / "models"
for _d in (OUT_DIR, SPLIT_DIR, FIG_DIR, RESULT_DIR, TABLE_DIR, MODEL_DIR):
    _d.mkdir(parents=True, exist_ok=True)

ROOT_DIR = BASE.resolve().parent

# Chemins dérivés propres à NB06.
COHORT_PATH = OUT_DIR / "cohort_nlp.csv"
TABLE_DIR   = RESULT_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if not COHORT_PATH.exists():
    raise FileNotFoundError(
        f"Cohorte introuvable : {COHORT_PATH}\n"
        "Exécuter d'abord NB01 puis NB02."
    )

df_desc = pd.read_csv(COHORT_PATH)

# Contrôle défensif avant le calcul des statistiques descriptives.
# Une valeur de douleur inférieure à 0 ou supérieure à 10 n'appartient pas
# à l'échelle numérique retenue et doit être traitée comme manquante.
if 'pain' in df_desc.columns:
    df_desc['pain'] = pd.to_numeric(df_desc['pain'], errors='coerce')
    pain_hors_plage = df_desc['pain'].notna() & ~df_desc['pain'].between(
        0, 10, inclusive='both'
    )
    n_pain_hors_plage = int(pain_hors_plage.sum())

    if n_pain_hors_plage:
        df_desc.loc[pain_hors_plage, 'pain'] = np.nan
        print(
            f'Douleur : {n_pain_hors_plage:,} valeurs hors 0–10 '
            'exclues des statistiques descriptives.'
        )

    assert df_desc['pain'].dropna().between(0, 10, inclusive='both').all(), (
        'Des valeurs de douleur hors de l’échelle 0–10 subsistent.'
    )

# Suppression défensive des colonnes dupliquées si un export précédent en a créé.
if df_desc.columns.duplicated().any():
    dup = df_desc.columns[df_desc.columns.duplicated()].tolist()
    print(f"⚠ Colonnes dupliquées détectées et retirées : {dup}")
    df_desc = df_desc.loc[:, ~df_desc.columns.duplicated()].copy()

colonnes_source = list(df_desc.columns)

# Cible : compatibilité avec les différents noms observés dans les notebooks.
if "target" in df_desc.columns:
    TARGET = "target"
elif "hospitalization" in df_desc.columns:
    TARGET = "hospitalization"
else:
    raise ValueError("Aucune cible trouvée : ni 'target', ni 'hospitalization'.")

AGE_COL = "age_ed" if "age_ed" in df_desc.columns else ("anchor_age" if "anchor_age" in df_desc.columns else None)
CHARLSON_COL = "charlson_score" if "charlson_score" in df_desc.columns else (
    "charlson_score_simplifie" if "charlson_score_simplifie" in df_desc.columns else None
)

if "subject_id" not in df_desc.columns:
    raise ValueError("La colonne 'subject_id' est indispensable pour compter les patients uniques.")

#print(f"Racine projet : {ROOT_DIR}")
#print(f"Cohorte chargée : {COHORT_PATH}")
print(f"Dimensions : {df_desc.shape[0]:,} lignes × {df_desc.shape[1]:,} colonnes")
print(f"Cible utilisée : {TARGET}")
print(f"Variable d'âge utilisée : {AGE_COL}")
print(f"Variable Charlson utilisée : {CHARLSON_COL}")
print(f"Nombre de variables sources intégrées dans la table exhaustive : {len(colonnes_source)}")


Dimensions : 383,919 lignes × 38 colonnes
Cible utilisée : target
Variable d'âge utilisée : age_ed
Variable Charlson utilisée : charlson_score
Nombre de variables sources intégrées dans la table exhaustive : 38


In [2]:
# Préparation de quelques variables dérivées pour la Table 1 
# Ces variables servent uniquement à la présentation. Elles ne remplacent pas les colonnes sources.

if AGE_COL is not None:
    df_desc["age_groupe"] = pd.cut(
        pd.to_numeric(df_desc[AGE_COL], errors="coerce"),
        bins=[17, 44, 64, 74, 200],
        labels=["18–44 ans", "45–64 ans", "65–74 ans", "≥75 ans"]
    )
else:
    df_desc["age_groupe"] = pd.NA

df_desc["sortie"] = df_desc[TARGET].map({
    0: "Retour à domicile",
    1: "Hospitalisation"
}).fillna("Issue non renseignée")

for col in ["gender", "anchor_year_group", "arrival_transport", "disposition"]:
    if col in df_desc.columns:
        df_desc[col] = df_desc[col].fillna("Non renseigné")

print("Variables de présentation préparées.")


Variables de présentation préparées.


In [3]:
# Fonctions de format revue scientifique 
# IC95 moyenne : moyenne ± 1,96 erreur standard
# IC95 proportion : intervalle de Wilson

def fmt_n(n):
    return f"{int(n):,}".replace(",", " ")

def fmt_num(x, digits=1):
    if pd.isna(x):
        return ""
    x = round(float(x), digits)
    return str(int(x)) if x.is_integer() else f"{x:.{digits}f}"

def fmt_pct(x):
    """Formatte un pourcentage sans masquer les très petites proportions.

    Objectif : éviter les affichages du type N(0 %) lorsque la proportion
    est faible mais non nulle. La précision augmente automatiquement sous 1 %.
    """
    if pd.isna(x):
        return ""
    x = float(x)
    if x == 0:
        return "0"
    ax = abs(x)
    if ax < 0.01:
        return f"{x:.4f}"
    if ax < 0.1:
        return f"{x:.3f}"
    if ax < 1:
        return f"{x:.2f}"
    return f"{x:.1f}"

def mean_sd_median_iqr_minmax(var):
    s = pd.to_numeric(df_desc[var], errors="coerce").dropna()
    if len(s) == 0:
        return "Non disponible"
    q1, q3 = s.quantile([0.25, 0.75])
    return (
        f"{fmt_num(s.mean())} ± {fmt_num(s.std())} ; "
        f"{fmt_num(s.median())} [{fmt_num(q1)}–{fmt_num(q3)}] ; "
        f"{fmt_num(s.min())}–{fmt_num(s.max())}"
    )

def ic95_mean(var):
    s = pd.to_numeric(df_desc[var], errors="coerce").dropna()
    n = len(s)
    if n == 0:
        return ""
    m = s.mean()
    se = s.std() / np.sqrt(n)
    return f"[{fmt_num(m - 1.96*se)}–{fmt_num(m + 1.96*se)}]"

def ic95_wilson(n, total):
    if total == 0:
        return ""
    z = 1.96
    p = n / total
    denom = 1 + z**2 / total
    centre = p + z**2 / (2 * total)
    marge = z * np.sqrt((p * (1 - p) + z**2 / (4 * total)) / total)
    low = (centre - marge) / denom
    high = (centre + marge) / denom
    return f"[{fmt_pct(low*100)}–{fmt_pct(high*100)} %]"

def n_pct(n, total):
    if total == 0:
        return "0 (0 %)"
    return f"{fmt_n(n)} ({fmt_pct(n / total * 100)} %)"


In [4]:
#  Table 1 : Format revue scientifique 
# Une ligne par caractéristique clinique principale, population totale uniquement.

rows = []
n_total = len(df_desc)

rows.append(["Nombre de séjours", fmt_n(n_total), ""])
rows.append(["Nombre de patients uniques", fmt_n(df_desc["subject_id"].nunique()), ""])

# Devenir
for modalite in ["Retour à domicile", "Hospitalisation"]:
    n = int((df_desc["sortie"] == modalite).sum())
    rows.append([f"{modalite}, n (%)", n_pct(n, n_total), ic95_wilson(n, n_total)])

# Âge
if AGE_COL is not None:
    rows.append([
        "Âge, moyenne ± SD ; médiane [IQR] ; min–max",
        mean_sd_median_iqr_minmax(AGE_COL),
        ic95_mean(AGE_COL)
    ])

    for modalite, n in df_desc["age_groupe"].value_counts().sort_index().items():
        rows.append([f"Âge {modalite}, n (%)", n_pct(int(n), n_total), ic95_wilson(int(n), n_total)])

# Sexe / genre
if "gender" in df_desc.columns:
    for modalite, n in df_desc["gender"].value_counts(dropna=False).sort_index().items():
        rows.append([f"Sexe/genre : {modalite}, n (%)", n_pct(int(n), n_total), ic95_wilson(int(n), n_total)])

# Transport d'arrivée
if "arrival_transport" in df_desc.columns:
    for modalite, n in df_desc["arrival_transport"].value_counts(dropna=False).sort_values(ascending=False).items():
        rows.append([f"Mode d'arrivée : {modalite}, n (%)", n_pct(int(n), n_total), ic95_wilson(int(n), n_total)])

# Acuity / ESI
if "acuity" in df_desc.columns:
    for modalite, n in df_desc["acuity"].value_counts(dropna=False).sort_index().items():
        rows.append([f"ESI / acuity {modalite}, n (%)", n_pct(int(n), n_total), ic95_wilson(int(n), n_total)])

# Constantes vitales et variables numériques cliniques fréquentes si présentes
variables_numeriques_principales = [
    ("temperature", "Température"),
    ("heartrate", "Fréquence cardiaque"),
    ("resprate", "Fréquence respiratoire"),
    ("o2sat", "Saturation O₂"),
    ("sbp", "Pression artérielle systolique"),
    ("dbp", "Pression artérielle diastolique"),
    ("pain", "Douleur"),
    ("pain_numeric", "Douleur numérique"),
    ("n_hosp_anterieures", "Nombre d'hospitalisations antérieures"),
]

for var, label in variables_numeriques_principales:
    if var in df_desc.columns:
        rows.append([
            f"{label}, moyenne ± SD ; médiane [IQR] ; min–max",
            mean_sd_median_iqr_minmax(var),
            ic95_mean(var)
        ])

# Charlson
if CHARLSON_COL is not None:
    rows.append([
        f"Score de Charlson ({CHARLSON_COL}), moyenne ± SD ; médiane [IQR] ; min–max",
        mean_sd_median_iqr_minmax(CHARLSON_COL),
        ic95_mean(CHARLSON_COL)
    ])

# Comorbidités : toutes les variables cm_* présentes dans cohort_nlp.csv
cm_cols = sorted([c for c in colonnes_source if c.startswith("cm_")])
for var in cm_cols:
    label = var.replace("cm_", "").replace("_", " ").capitalize()
    n = int((pd.to_numeric(df_desc[var], errors="coerce") == 1).sum())
    rows.append([f"{label}, n (%)", n_pct(n, n_total), ic95_wilson(n, n_total)])

table1_revue = pd.DataFrame(
    rows,
    columns=[
        "Caractéristique",
        f"Population totale (N = {fmt_n(n_total)})",
        "IC95"
    ]
)

display(
    table1_revue.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "left",
        "white-space": "pre-wrap"
    })
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left"), ("font-weight", "bold")]},
        {"selector": "td", "props": [("padding", "6px")]}
    ])
)


Caractéristique,Population totale (N = 383 919),IC95
Nombre de séjours,383 919,
Nombre de patients uniques,188 127,
"Retour à domicile, n (%)",237 104 (61.8 %),[61.6–61.9 %]
"Hospitalisation, n (%)",146 815 (38.2 %),[38.1–38.4 %]
"Âge, moyenne ± SD ; médiane [IQR] ; min–max",53.1 ± 20.6 ; 54 [35–69] ; 18–103,[53–53.1]
"Âge 18–44 ans, n (%)",138 720 (36.1 %),[36.0–36.3 %]
"Âge 45–64 ans, n (%)",122 911 (32.0 %),[31.9–32.2 %]
"Âge 65–74 ans, n (%)",55 011 (14.3 %),[14.2–14.4 %]
"Âge ≥75 ans, n (%)",67 277 (17.5 %),[17.4–17.6 %]
"Sexe/genre : F, n (%)",209 994 (54.7 %),[54.5–54.9 %]


In [5]:
#  Sauvegarde des tableaux 
out_table1_csv = TABLE_DIR / "tab39_table1_format_revue.csv"
out_table1_xlsx = TABLE_DIR / "tab39_table1_format_revue.xlsx"
out_all_csv = TABLE_DIR / "table_variables_completes_cohort_nlp.csv"
out_all_xlsx = TABLE_DIR / "table_variables_completes_cohort_nlp.xlsx"

table1_revue.to_csv(out_table1_csv, index=False)
table1_revue.to_excel(out_table1_xlsx, index=False)

#table_variables_completes.to_csv(out_all_csv, index=False)
#table_variables_completes.to_excel(out_all_xlsx, index=False)

print(f"Table 1 sauvegardée en csv : OK")
print(f"Table 1 sauvegardée en xlsx: OK")
print(f"Table exhaustive sauvegardée en csv: OK")
print(f"Table exhaustive sauvegardée en xlsx: OK")
print("Note : la table exhaustive contient toutes les variables sources de data/processed/cohort_nlp.csv.")


Table 1 sauvegardée en csv : OK
Table 1 sauvegardée en xlsx: OK
Table exhaustive sauvegardée en csv: OK
Table exhaustive sauvegardée en xlsx: OK
Note : la table exhaustive contient toutes les variables sources de data/processed/cohort_nlp.csv.


In [6]:
# Dossier de sauvegarde de la Table 1
FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Fichiers de la Table 1
out_table1_csv = FIGURE_DIR / "tab39_table1_format_revue.csv"
out_table1_xlsx = FIGURE_DIR / "tab39_table1_format_revue.xlsx"

# Sauvegarde de la Table 1
table1_revue.to_csv(out_table1_csv, index=False)
table1_revue.to_excel(out_table1_xlsx, index=False)

print(f"Table 1 sauvegardée en CSV : {out_table1_csv}")
print(f"Table 1 sauvegardée en XLSX : {out_table1_xlsx}")

Table 1 sauvegardée en CSV : ..\figures\tab39_table1_format_revue.csv
Table 1 sauvegardée en XLSX : ..\figures\tab39_table1_format_revue.xlsx


In [7]:
from pathlib import Path
import matplotlib.pyplot as plt
import textwrap

# Dossier de sauvegarde
FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Fichiers de sortie
out_table1_csv = FIGURE_DIR / "table1_format_revue.csv"
out_table1_xlsx = FIGURE_DIR / "table1_format_revue.xlsx"
out_table1_png = FIGURE_DIR / "table1_format_revue.png"

# Sauvegarde en CSV et Excel
table1_revue.to_csv(
    out_table1_csv,
    index=False,
    encoding="utf-8-sig"
)

table1_revue.to_excel(
    out_table1_xlsx,
    index=False
)

# Préparation de la table pour l'image
table1_word = table1_revue.copy()

# Retour à la ligne automatique dans la colonne Caractéristique
if "Caractéristique" in table1_word.columns:
    table1_word["Caractéristique"] = (
        table1_word["Caractéristique"]
        .astype(str)
        .apply(lambda texte: "\n".join(textwrap.wrap(texte, width=48)))
    )

# Hauteur adaptée au nombre de lignes
hauteur_figure = max(5, 0.48 * (len(table1_word) + 1))

fig, ax = plt.subplots(
    figsize=(14, hauteur_figure)
)

ax.axis("off")

table_image = ax.table(
    cellText=table1_word.values,
    colLabels=table1_word.columns,
    cellLoc="left",
    colLoc="left",
    loc="center"
)

# Mise en forme
table_image.auto_set_font_size(False)
table_image.set_fontsize(9)
table_image.scale(1, 1.45)

# En-têtes en gras et largeur des colonnes
for (ligne, colonne), cellule in table_image.get_celld().items():

    if ligne == 0:
        cellule.set_text_props(weight="bold")

    if colonne == 0:
        cellule.set_width(0.55)
    elif colonne == 1:
        cellule.set_width(0.30)
    else:
        cellule.set_width(0.15)

plt.tight_layout()

# Image haute résolution pour Word
fig.savefig(
    out_table1_png,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print(f"Table 1 sauvegardée en CSV : {out_table1_csv}")
print(f"Table 1 sauvegardée en Excel : {out_table1_xlsx}")
print(f"Table 1 prête pour Word : {out_table1_png}")

Table 1 sauvegardée en CSV : ..\figures\table1_format_revue.csv
Table 1 sauvegardée en Excel : ..\figures\table1_format_revue.xlsx
Table 1 prête pour Word : ..\figures\table1_format_revue.png
